# Pandas — Condensed Practical Notes
**Focus:** data analysis, ML preprocessing, and interview-relevant operations.  
Use this as the primary revision notebook; the original contains deeper explanations and extended exercises.

## 1. Setup & mental model
Pandas works with labeled tabular data: a **Series** is 1D; a **DataFrame** is 2D. Labels align during many operations.

In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    "name": ["Aman", "Riya", "Kabir", "Sara"],
    "dept": ["Tech", "HR", "Tech", "Sales"],
    "salary": [90000, 65000, 110000, 72000],
    "experience": [2, 4, 6, 3]
})

## 2. Series essentials
Create a Series, inspect labels/values, use vectorized operations, and remember arithmetic aligns by index.

In [ ]:
marks = pd.Series([78, 91, 84], index=["Math", "CS", "Physics"])
marks["CS"], marks.mean(), marks[marks >= 85]

In [ ]:
a = pd.Series([10, 20], index=["x", "y"])
b = pd.Series([1, 2], index=["y", "z"])
a + b  # aligns labels; unmatched labels become NaN

## 3. Inspecting & selecting data
Know `head`, `shape`, `columns`, `dtypes`, `info`, and `describe`. Prefer `.loc` for labels/conditions and `.iloc` for integer positions.

In [ ]:
df.head()
df.shape, df.columns, df.dtypes

In [ ]:
df.loc[df["salary"] > 70000, ["name", "salary"]]

In [ ]:
df.iloc[:3, :2]  # first 3 rows, first 2 columns

## 4. Filtering rows
Use `&`, `|`, and `~` with parentheses. `isin` and `between` keep filters readable.

In [ ]:
df[(df["dept"].isin(["Tech", "Sales"])) & (df["salary"].between(70000, 100000))]

## 5. Create, modify, sort columns
Vectorized expressions are usually clearer and faster than row-wise loops. Use `assign` for chainable transformations.

In [ ]:
df = df.assign(
    salary_lakh=df["salary"] / 100000,
    senior=df["experience"] >= 5
)
df.sort_values(["salary", "experience"], ascending=[False, True])

In [ ]:
df["level"] = np.where(df["experience"] >= 5, "Senior", "Junior")
df.nlargest(2, "salary")

## 6. Missing values & duplicates
Use `isna`/`notna`; choose dropping or filling based on meaning. Avoid comparing directly with `NaN`.

In [ ]:
df.isna().sum()
df.dropna(subset=["salary"])
df["salary"] = df["salary"].fillna(df["salary"].median())
df = df.drop_duplicates()

## 7. GroupBy — high value for analytics
`agg` summarizes groups; `transform` returns a group-level result aligned to every original row.

In [ ]:
df.groupby("dept").agg(
    headcount=("name", "count"),
    avg_salary=("salary", "mean"),
    median_salary=("salary", "median")
).sort_values("avg_salary", ascending=False)

In [ ]:
df["dept_avg"] = df.groupby("dept")["salary"].transform("mean")
df[["name", "dept", "salary", "dept_avg"]]

## 8. Text & dates
Use the `.str` and `.dt` accessors for vectorized operations.

In [ ]:
names = df["name"].str.strip().str.lower()

In [ ]:
dates = pd.to_datetime(pd.Series(["2025-01-10", "2025-03-15"]), errors="coerce")
dates.dt.year

## 9. Combine & reshape data
Use `merge` to join tables, `concat` to stack them, `pivot_table` for grouped summaries, and `melt` to go wide → long. Validate joins when possible.

In [ ]:
dept_info = pd.DataFrame({"dept": ["Tech", "HR", "Sales"], "budget": [500000, 150000, 200000]})
joined = df.merge(dept_info, on="dept", how="left", validate="many_to_one")

In [ ]:
pd.concat([df, df], ignore_index=True).head()
df.pivot_table(index="dept", values="salary", aggfunc="median")

## 10. Read/write & cleaning pattern

In [ ]:
# Common I/O
# data = pd.read_csv("file.csv", parse_dates=["date"], na_values=["-", "NA"])
# data.to_csv("cleaned.csv", index=False)

def clean_table(raw):
    out = raw.copy()
    out.columns = out.columns.str.strip().str.lower().str.replace(r"\W+", "_", regex=True)
    out = out.drop_duplicates()
    return out

## 11. Common pitfalls — quick checklist
- Use `.loc[mask, "col"] = value` rather than chained assignment.
- Assign results from methods such as `sort_values`, `drop`, and `fillna`.
- Use `.isna()` to detect missing values.
- Use `&`/`|` (with parentheses), not Python `and`/`or`, for Series filters.
- Check row counts after merges; use `validate=` where appropriate.
- `groupby` may sort group keys; set `sort=False` if order matters.

## Quick reference
| Task | Pattern |
|---|---|
| Read CSV | `pd.read_csv(path)` |
| Inspect | `df.shape`, `df.info()`, `df.describe()` |
| Select | `df["col"]`, `df.loc[rows, cols]`, `df.iloc[i, j]` |
| Filter | `df[(cond1) & (cond2)]` |
| Missing | `df.isna().sum()`, `df.fillna(value)` |
| Group | `df.groupby(key).agg(...)` |
| Per-row group statistic | `df.groupby(key)[col].transform("mean")` |
| Join | `left.merge(right, on=key, how="left", validate=...)` |
| Dates | `pd.to_datetime(s, errors="coerce")` |
| Text | `s.str.strip()`, `s.str.lower()` |